In [35]:
!date

import zarr
import os
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

Thu Mar 12 19:14:57 GMT 2026


In [33]:
PATH_TO_READ_COUNTS = "/lustre/scratch127/gsu/malariagen/production/runs/cnv/readcounts"
CONTIGS = [
    "Pf3D7_01_v3", "Pf3D7_02_v3", "Pf3D7_03_v3", "Pf3D7_04_v3",
    "Pf3D7_05_v3", "Pf3D7_06_v3", "Pf3D7_07_v3", "Pf3D7_08_v3",
    "Pf3D7_09_v3", "Pf3D7_10_v3", "Pf3D7_11_v3", "Pf3D7_12_v3",
    "Pf3D7_13_v3", "Pf3D7_14_v3", "Pf3D7_API_v3", "Pf3D7_MIT_v3"
]
CONTIG_TO_IDX = {c: i for i, c in enumerate(CONTIGS)}

def read_counts_tsv(path: str) -> pd.DataFrame:
    """Read a GATK CollectReadCounts TSV, skipping the SAM header lines."""
    with open(path) as f:
        skip = sum(1 for line in f if line.startswith("@"))
    df = pd.read_csv(path, sep="\t", skiprows=skip)
    df = df[df["CONTIG"].isin(CONTIG_TO_IDX)]
    df["POS"] = ((df["START"] + df["END"]) / 2).astype(np.uint32)
    df["CONTIG_IDX"] = df["CONTIG"].map(CONTIG_TO_IDX).astype(np.uint8)
    return df[["CONTIG_IDX", "POS", "COUNT"]]

def initialise_store(store_path: str, tsv_path: str, n_samples_estimate: int = 60000):
    """
    Create the zarr store using the first TSV to define coordinates.
    Only needs to be called once.
    """
    df = read_counts_tsv(tsv_path)
    n_bins = len(df)
    store = zarr.open(store_path, mode="w")
    store.array("contig_idx", df["CONTIG_IDX"].values, dtype="u1", chunks=False)
    store.array("pos",        df["POS"].values,        dtype="u4", chunks=False)
    store.zeros(
        "counts",
        shape=(0, n_bins),
        chunks=(100, n_bins),
        dtype="u4",
        compressor=zarr.Blosc(cname="lz4", clevel=5, shuffle=zarr.Blosc.SHUFFLE),
    )
    store.attrs.update({
        "contig_names": CONTIGS,
        "bin_size":     100,
        "genome":       "PlasmoDB-54_Pfalciparum3D7",
        "n_bins":       n_bins,
        "sample_ids":   [],
    })
    print(f"Initialised store: {n_bins} bins, expecting ~{n_samples_estimate} samples")
    return store

def append_sample(store_path: str, tsv_path: str, sample_id: str):
    """Append one sample to an existing store."""
    store  = zarr.open(store_path, mode="a")
    df     = read_counts_tsv(tsv_path)
    counts = store["counts"]
    counts.resize(counts.shape[0] + 1, counts.shape[1])
    counts[-1, :] = df["COUNT"].values.astype(np.uint16)
    attrs = store.attrs.asdict()
    attrs["sample_ids"].append(sample_id)
    store.attrs.put(attrs)

def bulk_append(store_path: str, tsv_dir: str, sample_ids: list[str], n_workers: int = 8):
    store  = zarr.open(store_path, mode="a")
    counts = store["counts"]
    attrs  = store.attrs.asdict()

    already_present = set(attrs["sample_ids"])
    to_add = [s for s in sample_ids if s not in already_present]

    if not to_add:
        print("No new samples to append.")
        return

    n_existing = counts.shape[0]
    n_new      = len(to_add)
    n_bins     = counts.shape[1]

    # Pre-resize once before any parallel writes
    counts.resize(n_existing + n_new, n_bins)

    def read_one(args):
        idx, sample_id = args
        tsv_path = os.path.join(tsv_dir, f"{sample_id}.counts.tsv")
        try:
            df = read_counts_tsv(tsv_path)
            assert len(df) == n_bins, f"Bin count mismatch: expected {n_bins}, got {len(df)}"
            return idx, sample_id, df["COUNT"].values.astype(np.uint16)
        except Exception as e:
            print(f"Failed to read {sample_id}: {e}")
            return idx, sample_id, None

    failed = []
    jobs   = [(n_existing + i, sid) for i, sid in enumerate(to_add)]

    with ThreadPoolExecutor(max_workers=n_workers) as pool:
        for idx, sample_id, data in tqdm(pool.map(read_one, jobs), total=len(jobs)):
            if data is not None:
                counts[idx, :] = data
                attrs["sample_ids"].append(sample_id)
            else:
                failed.append(sample_id)

    store.attrs.put(attrs)
    print(f"Appended {n_new - len(failed)}/{n_new} samples. Failed: {len(failed)}")

In [ ]:
ZARR_OUTPUT_DIR = "../../../data/raw/Pf9-53973-samples-100bp.zarr"

# 1. Initialise the store once, using any one TSV to define the bin coordinates
initialise_store(
    store_path = ZARR_OUTPUT_DIR,
    tsv_path   = "/lustre/scratch127/gsu/malariagen/production/runs/cnv/readcounts/FP0008-C.counts.tsv",
)

# 2. Get all sample IDs from your readcounts directory
tsv_files  = sorted(f for f in os.listdir(PATH_TO_READ_COUNTS) if f.endswith(".counts.tsv"))
sample_ids = [f.replace(".counts.tsv", "") for f in tsv_files]

# 3. Bulk append all samples
bulk_append(
    store_path = ZARR_OUTPUT_DIR,
    tsv_dir    = PATH_TO_READ_COUNTS,
    sample_ids = sample_ids,
)

# 4. Sanity check
store = zarr.open(ZARR_OUTPUT_DIR, mode="r")
print(store["counts"].shape)          # should be (n_samples, n_bins)
print(len(store.attrs["sample_ids"])) # should match n_samples
print(store["counts"][0, :20])        # first 20 bins of first sample

Initialised store: 233337 bins, expecting ~60000 samples


  0%|          | 0/53973 [00:00<?, ?it/s]